In [1]:
# Install necessary libraries
!pip install --upgrade -q gspread scikit-learn pandas

In [2]:
#Google Sheets Linking (Needs user authorization)
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
creds, _ = default()

gc = gspread.authorize(creds)

In [3]:
import pandas as pd

# Open the spreadsheet by URL
spreadsheet_url = 'https://docs.google.com/spreadsheets/d/14pMQzQwy2Dl9OIhS24hyJdX1sfEdVi9SeZfGuYaMKQ0/edit?usp=drive_link'
wb = gc.open_by_url(spreadsheet_url)
sheet = wb.get_worksheet(0)

# Load data into DataFrame
data = sheet.get_all_values()
df = pd.DataFrame(data[1:], columns=data[0])

In [ ]:
#Imports
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.naive_bayes import MultinomialNB
import pickle


In [ ]:


# ------------------------PART 1: GENERIC LABEL MODEL (Binary Classification)
# Task: phishing vs legit

print("Training Naive Bayes Model (Generic Labels)...")

#--------Convert text labels into binary values:------
# phishing = 1, legit = 0
y_generic = df['label_generic'].apply(lambda x: 1 if x == 'phishing' else 0)

# ----------Create a Pipeline:-------------------
# Step 1: TF-IDF vectorization
# Step 2: Multinomial Naive Bayes classifier
# Using a Pipeline prevents data leakage during cross-validation
pipe_nb_generic = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1,2),        # use unigrams + bigrams
        stop_words='english',     # remove common English words
        max_features=10000        # limit vocabulary size
    )),
    ('nb', MultinomialNB())       # Naive Bayes classifier
])

# ----------Hyperparameter grid for tuning--------
# alpha = smoothing parameter
# fit_prior = whether to learn class prior probabilities
param_grid_generic = {
    'nb__alpha': [0.01, 0.1, 0.5, 1.0, 2.0],
    'nb__fit_prior': [True, False]
}

# Stratified K-Fold ensures class distribution is preserved in each fold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

#----------GridSearchCV:-------------
# - Tests all combinations of hyperparameters
# - Uses cross-validation
# - Selects the best model based on F1 score
grid_nb_generic = GridSearchCV(
    pipe_nb_generic,
    param_grid=param_grid_generic,
    cv=skf,
    scoring='f1',        # good metric for binary phishing detection
    verbose=1
)
#-------- Train + tune model-----------
grid_nb_generic.fit(df['text'].astype(str), y_generic)

print("Best Generic Parameters:", grid_nb_generic.best_params_)
print("Best Generic F1 Score:", grid_nb_generic.best_score_)

# ---------Save trained model--------
NB_Model_generic = grid_nb_generic.best_estimator_

with open("naive_bayes_generic.pkl", "wb") as f:
    pickle.dump(NB_Model_generic, f)

print("Saved: naive_bayes_generic.pkl")

In [5]:


# -----------------PART 2: SPECIFIC LABEL MODEL (Multiclass Classification)
# Task: human_legit, human_phishing, llm_legit, llm_phishing



print("\nTraining Naive Bayes Model (Specific Labels)...")

#--------- Encode 4 text labels into numerical values
le = LabelEncoder()
y_specific = le.fit_transform(df['label_specific'])

pipe_nb_specific = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1,2),
        stop_words='english',
        max_features=10000
    )),
    ('nb', MultinomialNB())
])

#------------ Slightly different hyperparameter grid
param_grid_specific = {
    'nb__alpha': [0.1, 0.5, 1.0, 2.0, 3.0],
    'nb__fit_prior': [True, False]
}

#---------- GridSearch with macro F1 since this is multiclass
grid_nb_specific = GridSearchCV(
    pipe_nb_specific,
    param_grid=param_grid_specific,
    cv=skf,
    scoring='f1_macro',   # balances performance across all 4 classes
    verbose=1
)



grid_nb_specific.fit(df['text'].astype(str), y_specific)

print("Best Specific Parameters:", grid_nb_specific.best_params_)
print("Best Specific F1-Macro Score:", grid_nb_specific.best_score_)

NB_Model_specific = grid_nb_specific.best_estimator_

#----------- Save model and label encoder
with open("naive_bayes_specific.pkl", "wb") as f:
    pickle.dump(NB_Model_specific, f)

with open("label_encoder_specific.pkl", "wb") as f:
    pickle.dump(le, f)

print("Saved: naive_bayes_specific.pkl")
print("Saved: label_encoder_specific.pkl")




Training Naive Bayes Model (Generic Labels)...
Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best Generic Parameters: {'nb__alpha': 0.01, 'nb__fit_prior': True}
Best Generic F1 Score: 0.984001307585989
Saved: naive_bayes_generic.pkl

Training Naive Bayes Model (Specific Labels)...
Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best Specific Parameters: {'nb__alpha': 0.1, 'nb__fit_prior': True}
Best Specific F1-Macro Score: 0.9865008120361665
Saved: naive_bayes_specific.pkl
Saved: label_encoder_specific.pkl
Evaluation Data Loaded


,text,label_specific,label_generic
0,ClamAV database updated (09 Feb 2008 12-08 +00...,human_legit,legit
1,"From a trusted sender. Dear jose@monkey.org, P...",human_phishing,phishing
2,"There were a good number of skipped threads, b...",human_legit,legit
3,Message generated from monkey.org source. ...,human_phishing,phishing
4,-----BEGIN PGP SIGNED MESSAGE-----\nHash: SHA1...,human_legit,legit


Evaluation Predictions Complete


In [ ]:

# -------------------- LOAD + EVALUATE ON EVAL SET
# Load evaluation data
eval_url = "https://docs.google.com/spreadsheets/d/1KsidfWCrywomIAbdlNg6Gn5rMER9E8gpI2AxGpNfjZ4/edit?gid=1833299105#gid=1833299105"

wb_eval = gc.open_by_url(eval_url)
sheet_eval = wb_eval.get_worksheet(0)

eval_data = sheet_eval.get_all_values()
eval_df = pd.DataFrame(eval_data[1:], columns=eval_data[0])

print("Evaluation Data Loaded")
display(eval_df.head())

# Clean strings (optional but helpful)
eval_df["text"] = eval_df["text"].astype(str)
eval_df["label_generic"] = eval_df["label_generic"].astype(str).str.strip()
eval_df["label_specific"] = eval_df["label_specific"].astype(str).str.strip()

# ---------- GENERIC PREDICTIONS ----------
y_true_generic = eval_df["label_generic"].apply(lambda x: 1 if x == "phishing" else 0)
y_pred_generic = NB_Model_generic.predict(eval_df["text"])

# ---------- SPECIFIC PREDICTIONS ----------
y_true_specific = le.transform(eval_df["label_specific"])
y_pred_specific = NB_Model_specific.predict(eval_df["text"])

print("Evaluation Predictions Complete")

In [6]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import pandas as pd

#---------------REPORTS
print("\n=== EVAL REPORT: GENERIC ===")
print(classification_report(y_true_generic, y_pred_generic, target_names=["legit", "phishing"]))

print("\n=== EVAL REPORT: SPECIFIC ===")
print(classification_report(y_true_specific, y_pred_specific, target_names=le.classes_))

#
#------------------CONFUSION MATRICES
#
cm_g = confusion_matrix(y_true_generic, y_pred_generic)


cm_s = confusion_matrix(y_true_specific, y_pred_specific)

print("\nGeneric Confusion Matrix (labeled):")
cm_g_df = pd.DataFrame(cm_g, index=["Actual legit", "Actual phishing"],
                       columns=["Pred legit", "Pred phishing"])
display(cm_g_df)

print("\nSpecific Confusion Matrix (labeled):")
cm_s_df = pd.DataFrame(cm_s, index=le.classes_, columns=le.classes_)
display(cm_s_df)

print("Evaluation Predictions + Visualizations Complete")


=== EVAL REPORT: GENERIC ===
              precision    recall  f1-score   support

       legit       1.00      0.99      0.99       400
    phishing       0.99      1.00      1.00       400

    accuracy                           0.99       800
   macro avg       1.00      1.00      0.99       800
weighted avg       1.00      0.99      0.99       800


=== EVAL REPORT: SPECIFIC ===
                precision    recall  f1-score   support

   human_legit       1.00      1.00      1.00       200
human_phishing       1.00      1.00      1.00       200
     llm_legit       1.00      0.99      1.00       200
  llm_phishing       1.00      1.00      1.00       200

      accuracy                           1.00       800
     macro avg       1.00      1.00      1.00       800
  weighted avg       1.00      1.00      1.00       800


Generic Confusion Matrix (labeled):


,Pred legit,Pred phishing
Actual legit,397,3
Actual phishing,1,399



Specific Confusion Matrix (labeled):


,human_legit,human_phishing,llm_legit,llm_phishing
human_legit,200,0,0,0
human_phishing,0,200,0,0
llm_legit,0,0,199,1
llm_phishing,0,0,0,200


Evaluation Predictions + Visualizations Complete


In [7]:
import numpy as np

# Use the tuned specific model
tfidf_vec = NB_Model_specific.named_steps["tfidf"]
nb_clf = NB_Model_specific.named_steps["nb"]

feature_names = tfidf_vec.get_feature_names_out()

print("\nTop words per class (TUNED specific model):")
for i, label in enumerate(le.classes_):
    top_idx = np.argsort(nb_clf.feature_log_prob_[i])[-20:][::-1]
    print(f"\nTop words for {label}:")
    print([feature_names[j] for j in top_idx])


Top words per class (TUNED specific model):

Top words for human_legit:
['http', 'wrote', 'spambayes', 'mail', 'org', '2008', 'submission', 'list', 'www', 'python', 'id', 'http www', 'perl', 'py', '2007', 'use', 'submission id', 'self', 'spamassassin', '10']

Top words for human_phishing:
['monkey', 'monkey org', 'jose', 'org', 'jose monkey', 'account', 'email', 'messages', 'update', 'mail', '2022', 'password', 'message', 'emails', 'mailbox', '2021', 'verify', '2020', 'information', 'continue']

Top words for llm_legit:
['account', 'security', 'finds', 'regards', 'dear', 'https', 'email finds', 'com', 'hope', 'refund', 'prize', 'sarah', 'community', 'thank', 'email', 'hope email', 'warm regards', 'warm', 'click', 'tax']

Top words for llm_phishing:
['account', 'link', 'https', 'com', 'opportunity', 'dear', 'refund', 'claim', 'following link', 'security', 'regards', 'hope', 'support', 'secure', 'link https', 'following', 'click', 'hope email', 'link com', 'verification']
